In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv('Project Data/lending_club_train.csv')
test_df = pd.read_csv('Project Data/lending_club_test.csv')

In [3]:
print("--- Train Data Head ---")
display(train_df.head())

print("\n--- Test Data Head ---")
display(test_df.head())

--- Train Data Head ---


,issue_d,sub_grade,term,home_ownership,fico_range_low,total_acc,pub_rec,revol_util,annual_inc,int_rate,...,pub_rec_bankruptcies,addr_state,initial_list_status,fico_range_high,revol_bal,id,open_acc,emp_length,loan_status,time_to_earliest_cr_line
0,2014-01-01,D1,60 months,MORTGAGE,660.0,18.0,0.0,86.8,40440.0,16.99,...,0.0,AR,f,664.0,17264.0,11024793,11.0,2.0,1,478656.0
1,2014-01-01,C4,60 months,MORTGAGE,740.0,26.0,0.0,103.5,59000.0,15.61,...,0.0,VT,f,744.0,6725.0,10596078,8.0,3.0,1,541728.0
2,2014-01-01,A4,36 months,RENT,700.0,47.0,0.0,11.4,40000.0,7.90,...,0.0,TX,w,704.0,7613.0,10775616,27.0,1.0,1,657590.4
3,2014-01-01,D1,60 months,MORTGAGE,665.0,26.0,0.0,56.2,70000.0,16.99,...,0.0,IN,f,669.0,14173.0,10765610,9.0,11.0,1,328838.4
4,2014-01-01,C3,36 months,MORTGAGE,700.0,15.0,1.0,67.1,120000.0,14.98,...,0.0,LA,f,704.0,2549.0,10794837,8.0,2.0,1,305164.8



--- Test Data Head ---


,issue_d,sub_grade,term,home_ownership,fico_range_low,total_acc,pub_rec,revol_util,annual_inc,int_rate,...,pub_rec_bankruptcies,addr_state,initial_list_status,fico_range_high,revol_bal,id,open_acc,emp_length,loan_status,time_to_earliest_cr_line
0,2016-07-01,A4,36 months,MORTGAGE,830.0,13.0,0.0,12.0,105682.0,7.99,...,0.0,TX,w,834.0,4266.0,84759443,5.0,11.0,1.0,789004.8
1,2016-07-01,B5,36 months,RENT,660.0,25.0,0.0,59.4,68000.0,11.49,...,0.0,CA,w,664.0,6944.0,84433407,11.0,6.0,1.0,294624.0
2,2016-07-01,D2,36 months,MORTGAGE,660.0,17.0,1.0,40.9,75000.0,17.99,...,1.0,AZ,w,664.0,5528.0,84646007,6.0,3.0,1.0,349747.2
3,2016-07-01,A1,36 months,RENT,740.0,36.0,0.0,27.7,114000.0,5.32,...,0.0,CA,w,744.0,25912.0,84477434,16.0,1.0,1.0,691632.0
4,2016-07-01,C4,60 months,MORTGAGE,680.0,14.0,0.0,44.3,47000.0,14.49,...,0.0,OH,w,684.0,4205.0,84525798,12.0,2.0,0.0,591667.2


In [4]:
# Converting "issue_d" to datetime format
train_df['issue_d_dt'] = pd.to_datetime(train_df['issue_d'], errors='coerce')

In [5]:
# Filtering for 2016 data
train_2016 = train_df[train_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original training rows: {train_df.shape[0]}")
print(f"Filtered 2016 training rows: {train_2016.shape[0]}")
print(f"Months included in 2016 train set: {sorted(train_2016['issue_d_dt'].dt.month.unique())}")

Original training rows: 236846
Filtered 2016 training rows: 53844
Months included in 2016 train set: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


In [6]:
# Doing the same for the test data
test_df['issue_d_dt'] = pd.to_datetime(test_df['issue_d'], errors='coerce')
test_2016 = test_df[test_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original test rows: {test_df.shape[0]}")
print(f"Filtered 2016 test rows: {test_2016.shape[0]}")
print(f"Months included in 2016 test set: {sorted(test_2016['issue_d_dt'].dt.month.unique())}")

Original test rows: 95019
Filtered 2016 test rows: 31333
Months included in 2016 test set: [np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]


In [7]:
# Checking for nulls
print("--- TRAIN NULL % ---")
train_nulls = (train_2016.isnull().sum() / len(train_2016) * 100).round(2)
print(train_nulls[train_nulls > 0].sort_values(ascending=False))

print("\n--- TEST NULL % ---")
test_nulls = (test_2016.isnull().sum() / len(test_2016) * 100).round(2)
print(test_nulls[test_nulls > 0].sort_values(ascending=False))

--- TRAIN NULL % ---
Series([], dtype: float64)

--- TEST NULL % ---
Series([], dtype: float64)


In [8]:
# 1. Define columns to drop (IDs, dates, and data leakage columns)
cols_to_drop = ['id', 'issue_d', 'issue_d_dt', 'loan_status']

# 2. Separate the target variable (int_rate) from the feature sets
train_y = train_2016['int_rate']
test_y = test_2016['int_rate']

train_features = train_2016.drop(columns=cols_to_drop + ['int_rate'])
test_features = test_2016.drop(columns=cols_to_drop + ['int_rate'])

# 3. Combine temporarily to ensure One-Hot Encoding creates identical columns for both
combined_features = pd.concat([train_features, test_features], axis=0)
combined_encoded = pd.get_dummies(combined_features, drop_first=True)

# 4. Split them back apart into train and test sets
train_encoded = combined_encoded.iloc[:len(train_2016)]
test_encoded = combined_encoded.iloc[len(train_2016):]

# Quick check on the final shapes
print("Train features shape:", train_encoded.shape)
print("Test features shape:", test_encoded.shape)

Train features shape: (53844, 117)
Test features shape: (31333, 117)
